In [1]:
from typing import TypedDict, Annotated, Sequence

from langgraph.graph import StateGraph, START, END

from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage

from langchain_core.tools import tool

from langgraph.prebuilt import ToolNode

In [2]:
load_dotenv()

True

In [58]:
# Kind of makes sense to use BaseMessage as the type since all other message types extend it

class AgentState(TypedDict):
     messages: Annotated[list, add_messages]
    
@tool
def add(a:int, b:int) -> int:
    """An addition function that adds 2 numbers

    Args:
        a (int): first number
        b (int): second number
    
    Returns:
        int: sum of num a and num b
    """
    return a + b

@tool
def subtract(a:int, b:int) -> int:
    """An subtraction function that subtracts b from a

    Args:
        a (int): first number
        b (int): second number
    
    Returns:
        int: difference of num a and num b
    """
    return a - b

@tool
def multiply(a:int, b:int) -> int:
    """An multiplication function that multiplies 2 numbers

    Args:
        a (int): first number
        b (int): second number
    
    Returns:
        int: product of num a and num b
    """
    return a * b
tools = [add, subtract, multiply]

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview").bind_tools(tools)
    

In [ ]:
from typing import Literal


def model_call(state: AgentState) -> dict:
    
    system_prompt = SystemMessage(content= 'You are my AI assistant, please answer my query to the best of your knowledge')
    
    response = llm.invoke([system_prompt] + list(state['messages']))
    return {"messages": [response]}
    

def should_continue(state: AgentState) -> Literal['continue', 'end']: 
    messages = state['messages']
    last_message = messages[-1]
    if not last_message.tool_calls:
        return 'end'
    else:
        return 'continue'

In [60]:
graph = StateGraph(AgentState)

graph.add_node('agent', model_call)

tool_node = ToolNode(tools=tools)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'agent')
graph.add_conditional_edges(
    'agent',
    should_continue,
    {
        'continue': 'tools',
        'end': END
    }
)
graph.add_edge('tools', 'agent')

app = graph.compile()


In [61]:
def print_stream(stream):
    for s in stream:
        message = s['messages'][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [63]:
from typing import cast

from langchain_core.messages import HumanMessage

inputs: AgentState = cast(AgentState, {'messages': [HumanMessage(content='whats the sum of 21 and 7, then subtracted from 50. Also tell me a random fact')]})
print_stream(app.stream(inputs, stream_mode='values'))

================================ Human Message =================================

whats the sum of 21 and 7, then subtracted from 50. Also tell me a random fact
================================== Ai Message ==================================

[]
Tool Calls:
  add (3a738b59-3e33-4c55-8f24-aed18bbe5d7e)
 Call ID: 3a738b59-3e33-4c55-8f24-aed18bbe5d7e
  Args:
    a: 21
    b: 7
================================= Tool Message =================================
Name: add

28
================================== Ai Message ==================================

[]
Tool Calls:
  subtract (3698cc49-0156-43c3-bc90-d551040f697d)
 Call ID: 3698cc49-0156-43c3-bc90-d551040f697d
  Args:
    a: 50
    b: 28
================================= Tool Message =================================
Name: subtract

22
================================== Ai Message ==================================

[{'type': 'text', 'text': 'The sum of 21 and 7 is 28, and when you subtract that from 50, you get 22.\n\nHere is a random fa